### Context and structure of the notebook

This notebook covers data acquisition, database storage, cleaning and exploratory analysis of the FAOSTAT agricultural datasets used in the project.

* Loads FAOSTAT production, trade and land-use datasets
* Cleans column names and checks data structure, units, countries and years
* Reshapes long-format FAOSTAT data into wide country-year tables
* Stores prepared datasets in a MySQL database
* Merges production, trade and land-use data into one analytical dataset
* Engineers comparable indicators such as crop shares, export shares, land-use shares and production intensity measures
* Checks missing values, outliers, skewness and feature distributions
* Prepares the final structured dataset for machine learning and statistical analysis

In [85]:
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import numpy as np
import re

# Loads from .env to system variables
from dotenv import load_dotenv
# Loads from system variables to Python
from os import getenv
import sqlalchemy
import pymysql
from sqlalchemy import text as t

import warnings
warnings.filterwarnings('ignore') # supress the warnings

### FAOSTAT

*FAO LICENSES:*
- *FAO encourages you to use datasets contained in FAO corporate statistical databases for research, statistical, scientific and evidence-based decision-making purposes. You may access, download, create copies, adapt and re-disseminate datasets subject to these Database terms https://www.fao.org/contact-us/terms/db-terms-of-use/en/* *Unless specified otherwise in their metadata or webpage, all datasets disseminated through FAO corporate statistical databases are licensed under the Creative Commons Attribution-4.0 International licence (CC BY 4.0).*

Production dataset's metadata states that "For this particular data domain, data sourced from Eurostat, and the International Organisation of Vine and Wine (but we did not download wine or any wine-related data) and only marginally processed by FAO might be subject to additional restrictions". 

*EUROSTAT LICENSES:*
- *The copyright for the editorial content of this website, which is owned by the EU, is licensed under the Creative Commons Attribution 4.0 International licence. Reuse of statistical data, metadata, publications, and other dissemination tools published on this website for commercial or non-commercial purposes is authorised provided the source is acknowledged.*


Datasets:
- **Production** https://www.fao.org/faostat/en/#data/QCL
- **Trade** https://www.fao.org/faostat/en/#data/TCL
- **Land Use** https://www.fao.org/faostat/en/#data/RL

Countries: EU (27 countries), UK, USA, Canada, Australia, New Zealand, Argentina, Brazil, Uruguay, China, India, Thailand. Countries outside the EU were selected based on their strong livestock/dairy/crop/agricultural exporters profiles.

Years: 2000-2024



## Load FAOSTAT data and save it into MySQL

1. Load Crops and livestock production dataset

In [2]:
fao_prod_df = pd.read_csv("FAOSTAT_data_en_production.csv")
fao_prod_df.head()

,Domain Code,Domain,Area Code (M49),Area,Element Code,Element,Item Code (CPC),Item,Year Code,Year,Unit,Value,Flag,Flag Description,Note
0,QCL,Crops and livestock products,32,Argentina,5312,Area harvested,0115,Barley,2000,2000,ha,247830.0,A,Official figure,NaN
1,QCL,Crops and livestock products,32,Argentina,5412,Yield,0115,Barley,2000,2000,kg/ha,2915.3,A,Official figure,NaN
2,QCL,Crops and livestock products,32,Argentina,5510,Production,0115,Barley,2000,2000,t,722490.0,A,Official figure,NaN
3,QCL,Crops and livestock products,32,Argentina,5312,Area harvested,0115,Barley,2001,2001,ha,246490.0,A,Official figure,NaN
4,QCL,Crops and livestock products,32,Argentina,5412,Yield,0115,Barley,2001,2001,kg/ha,2150.4,A,Official figure,NaN


In [3]:
fao_prod_df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 48889 entries, 0 to 48888
Data columns (total 15 columns):
 #   Column            Non-Null Count  Dtype  
---  ------            --------------  -----  
 0   Domain Code       48889 non-null  object 
 1   Domain            48889 non-null  object 
 2   Area Code (M49)   48889 non-null  int64  
 3   Area              48889 non-null  object 
 4   Element Code      48889 non-null  int64  
 5   Element           48889 non-null  object 
 6   Item Code (CPC)   48889 non-null  object 
 7   Item              48889 non-null  object 
 8   Year Code         48889 non-null  int64  
 9   Year              48889 non-null  int64  
 10  Unit              48889 non-null  object 
 11  Value             48555 non-null  float64
 12  Flag              48889 non-null  object 
 13  Flag Description  48889 non-null  object 
 14  Note              529 non-null    object 
dtypes: float64(1), int64(4), object(10)
memory usage: 5.6+ MB


2. Check how many countries, years and production items we have:

In [311]:
# we do it by counting the length of unique elements numpy array
print(len(fao_prod_df["Area"].unique()), "countries")
print(len(fao_prod_df["Year"].unique()), "years")
print(len(fao_prod_df["Item"].unique()), "production items")

38 countries
25 years
30 production items


Also, some of the items, like Barley, have multiple measured elements (Area harvested, Yield, Production) and units (ha, kg/ha, t). 

We want to reshape the data so that:
- one row = one country + one year
- one column = one item + one element + one unit
- cell value = FAOSTAT Value

3. Check what combinations of Item + Element + Unit exist:

In [5]:
item_element_unit = fao_prod_df[["Item", "Element", "Unit"]].drop_duplicates().sort_values(["Item", "Element", "Unit"])
item_element_unit

,Item,Element,Unit
0,Barley,Area harvested,ha
2,Barley,Production,t
1,Barley,Yield,kg/ha
75,Butter of cow milk,Production,t
99,Cattle,Stocks,An
124,Cereals n.e.c.,Area harvested,ha
126,Cereals n.e.c.,Production,t
125,Cereals n.e.c.,Yield,kg/ha
2779,"Cheese from milk of goats, fresh or processed",Production,t
2803,"Cheese from milk of sheep, fresh or processed",Production,t


This shows the actual features that will be created.

We can also check how many final feature columns we will get:

In [6]:
item_element_unit.shape

(58, 3)

4. Create clean feature names

We need 'safe' column names, because FAOSTAT item names contain commas, slashes, brackets, spaces, etc.

In [7]:
def clean_name(text):
    text = str(text).lower()
    
    # make units clearer
    text = text.replace("/", "_per_")
    
    # replace non-alphanumeric characters with underscores
    text = re.sub(r"[^a-z0-9]+", "_", text)
    
    # remove duplicated underscores
    text = re.sub(r"_+", "_", text)
    
    # remove leading/trailing underscores
    text = text.strip("_")
    
    return text

We also need to limit the length of the feature name to avoid long column name problems with saving data to database. 

We can do that by explicitly manually truncating the longest item names (deleting last parts of the names that don't change the meaning and uniqueness of the item) and element names:

In [40]:
def simplify_item_name(item):
    item = str(item)

    replacements = {
        # " with the bone, fresh or chilled": "",
        ", fresh or chilled": "",
        ", fresh or processed": "",
        " n.e.c": "",
        ", fresh or dry": "",
        ", not carded or combed": "",
    }

    for old, new in replacements.items():
        item = item.replace(old, new)

    return item


def simplify_element_name(element):
    element = str(element)
    element = element.replace("Producing Animals/Slaughtered", "animals")
    replacements = {
        "Producing Animals/Slaughtered": "animals",
        "Production": "prod",
        "Area harvested": "area",
        "Yield/Carcass Weight": "weight",
        "Export quantity": "export_qty",
        "Export value": "export_value",
    }

    for old, new in replacements.items():
        element = element.replace(old, new)
    
    return element

Now create a feature column:

In [9]:
fao_prod_df["feature_name"] = (
    fao_prod_df["Item"].apply(simplify_item_name).apply(clean_name)
    + "_"
    + fao_prod_df["Element"].apply(simplify_element_name).apply(clean_name)
    + "_"
    + fao_prod_df["Unit"].apply(clean_name)
)

# check examples:
fao_prod_df[["Item", "Element", "Unit", "feature_name"]].drop_duplicates().sample(10)

,Item,Element,Unit,feature_name
873,Potatoes,Yield,kg/ha,potatoes_yield_kg_per_ha
1168,"Pulses, Total",Area harvested,ha,pulses_total_area_ha
2,Barley,Production,t,barley_prod_t
497,"Meat of chickens, fresh or chilled",Yield/Carcass Weight,g/An,meat_of_chickens_weight_g_per_an
723,"Meat of sheep, fresh or chilled",Production,t,meat_of_sheep_prod_t
199,Cheese from whole cow milk,Production,t,cheese_from_whole_cow_milk_prod_t
573,"Meat of goat, fresh or chilled",Production,t,meat_of_goat_prod_t
724,"Meat of sheep, fresh or chilled",Producing Animals/Slaughtered,An,meat_of_sheep_animals_an
2239,"Skim milk, condensed",Production,t,skim_milk_condensed_prod_t
498,"Meat of chickens, fresh or chilled",Production,t,meat_of_chickens_prod_t


5. Pivot the data frame to have features in colums

In [10]:
fao_production_df = fao_prod_df.pivot(
    index=["Area Code (M49)", "Area", "Year"],
    columns="feature_name",
    values="Value"
).reset_index()

# check the result:
fao_production_df.head()

feature_name,Area Code (M49),Area,Year,barley_area_ha,barley_prod_t,barley_yield_kg_per_ha,butter_of_cow_milk_prod_t,cattle_stocks_an,cereals_n_e_c_area_ha,cereals_n_e_c_prod_t,...,vegetables_primary_area_ha,vegetables_primary_prod_t,vegetables_primary_yield_kg_per_ha,wheat_area_ha,wheat_prod_t,wheat_yield_kg_per_ha,whey_condensed_prod_t,whey_dry_prod_t,whole_milk_condensed_prod_t,whole_milk_powder_prod_t
0,32,Argentina,2000,247830.0,722490.0,2915.3,52700.0,48674400.0,20000.0,13000.0,...,168152.0,2847168.58,16932.1,6408045.0,15959352.0,2490.5,NaN,2570.40,281.25,202000.0
1,32,Argentina,2001,246490.0,530045.0,2150.4,53400.0,48851400.0,20000.0,13000.0,...,163904.0,2854231.00,17414.0,6840720.0,15291660.0,2235.4,NaN,2639.12,135.00,185000.0
2,32,Argentina,2002,250760.0,549830.0,2192.7,54600.0,52000000.0,22000.0,15000.0,...,168147.0,2900791.13,17251.5,6050210.0,12301442.0,2033.2,NaN,5393.68,135.00,205000.0
3,32,Argentina,2003,334348.0,1005460.0,3007.2,55300.0,55875764.0,22000.0,15000.0,...,171193.0,2942954.60,17190.9,5735292.0,14562955.0,2539.2,NaN,5102.13,135.00,198000.0
4,32,Argentina,2004,275307.0,894610.0,3249.5,55000.0,56844020.0,22000.0,15000.0,...,170520.0,3017451.48,17695.5,6061930.0,15925025.0,2627.1,NaN,10710.00,135.00,220000.0


and shape:

In [11]:
fao_production_df.shape

(950, 61)

In [12]:
fao_production_df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 950 entries, 0 to 949
Data columns (total 61 columns):
 #   Column                                   Non-Null Count  Dtype  
---  ------                                   --------------  -----  
 0   Area Code (M49)                          950 non-null    int64  
 1   Area                                     950 non-null    object 
 2   Year                                     950 non-null    int64  
 3   barley_area_ha                           950 non-null    float64
 4   barley_prod_t                            950 non-null    float64
 5   barley_yield_kg_per_ha                   943 non-null    float64
 6   butter_of_cow_milk_prod_t                854 non-null    float64
 7   cattle_stocks_an                         950 non-null    float64
 8   cereals_n_e_c_area_ha                    543 non-null    float64
 9   cereals_n_e_c_prod_t                     534 non-null    float64
 10  cereals_n_e_c_yield_kg_per_ha            524 non-n

Rename ID columns for MySQL:

In [13]:
fao_production_df = fao_production_df.rename(columns = {"Area Code (M49)": "country_code",
                                                        "Area": "country_name",
                                                        "Year": "year"})

6. Store data frame to the database:

In [312]:
load_dotenv()

True

In [25]:
user = getenv("DB_USER")
password = getenv("PASSWORD")
host = getenv("HOST")
database = "agriculture"

In [315]:
sqlEngine = sqlalchemy.create_engine(f"mysql+pymysql://{user}:{password}@{host}")
conn = sqlEngine.connect()
conn.execute(t(f"CREATE DATABASE IF NOT EXISTS {database};"))

def exe(SQL_string):
    return conn.execute(t(SQL_string))

exe(f"USE {database};")

In [50]:
table_name = "faostat_crops_livestock_production"
fao_production_df.to_sql(table_name, conn, index=False, schema=database, if_exists="replace")
# save changes to data to sql server
conn.commit()

In [51]:
# check by querying Ireland data
pd.read_sql(
    """
    SELECT *
    FROM faostat_crops_livestock_production
    WHERE country_name = 'Ireland'
    ORDER BY year
    LIMIT 10;
    """,
    conn,
)

,country_code,country_name,year,barley_area_ha,barley_prod_t,barley_yield_kg_per_ha,butter_of_cow_milk_prod_t,cattle_stocks_an,cereals_n_e_c_area_ha,cereals_n_e_c_prod_t,...,vegetables_primary_area_ha,vegetables_primary_prod_t,vegetables_primary_yield_kg_per_ha,wheat_area_ha,wheat_prod_t,wheat_yield_kg_per_ha,whey_condensed_prod_t,whey_dry_prod_t,whole_milk_condensed_prod_t,whole_milk_powder_prod_t
0,372,Ireland,2000,182300.0,1309900.0,7185.4,145000.0,7037400.0,2000.0,3076.30,...,7463.0,232842.42,31198.0,78000.0,737400.0,9453.8,1200.0,34300.0,0.0,48000.0
1,372,Ireland,2001,182000.0,1277200.0,7017.6,139000.0,7049700.0,2200.0,3419.19,...,7933.0,250977.12,31635.7,84900.0,769200.0,9060.1,600.0,44100.0,0.0,39500.0
2,372,Ireland,2002,176000.0,962800.0,5470.5,147000.0,6992200.0,1800.0,2826.37,...,7464.0,232944.02,31207.7,102700.0,867200.0,8444.0,900.0,30100.0,0.0,31000.0
3,372,Ireland,2003,183100.0,1197700.0,6541.2,140100.0,6999500.0,3100.0,4917.32,...,7787.0,241713.08,31041.0,95700.0,794100.0,8297.8,1200.0,24500.0,0.0,39000.0
4,372,Ireland,2004,183700.0,1326600.0,7221.6,136900.0,7015600.0,3800.0,5965.89,...,8132.0,239202.03,29415.1,102700.0,1019200.0,9924.1,1200.0,35000.0,480.0,41000.0
5,372,Ireland,2005,164400.0,1024400.0,6231.1,143000.0,6991800.0,5500.0,7477.34,...,7353.0,247129.26,33608.0,95200.0,802700.0,8431.7,1800.0,36050.0,600.0,55700.0
6,372,Ireland,2006,167000.0,1136900.0,6807.8,139100.0,6977800.0,4900.0,6700.00,...,6971.0,231681.00,33234.9,87500.0,801000.0,9154.3,1800.0,44450.0,600.0,74000.0
7,372,Ireland,2007,167500.0,1124500.0,6713.4,141400.0,6890900.0,5800.0,8900.00,...,6731.0,223001.15,33129.6,84300.0,713400.0,8462.6,1800.0,69300.0,600.0,55700.0
8,372,Ireland,2008,187200.0,1294100.0,6912.9,123800.0,6902100.0,1300.0,2223.97,...,6864.0,218209.41,31790.3,110700.0,992800.0,8968.4,1800.0,44030.0,600.0,55000.0
9,372,Ireland,2009,193600.0,1227300.0,6339.4,120300.0,6890700.0,1300.0,2500.00,...,6505.0,210021.37,32287.6,84500.0,690100.0,8166.9,1800.0,37100.0,600.0,56000.0


#### Trade (export) data set

1. Load Crops and livestock trade datasets

This data is stored in two files: one has data for years 2000-2009, another for years 2010-2024.

In [162]:
trade_2000_2009 = pd.read_csv("FAOSTAT_data_en_trade_2000-2009.csv")
trade_2010_2024 = pd.read_csv("FAOSTAT_data_en_trade_2010-2024.csv")

Check shape of each data set anf years present:

In [163]:
print(trade_2000_2009.shape)
print(trade_2010_2024.shape)

print(trade_2000_2009["Year"].min(), trade_2000_2009["Year"].max())
print(trade_2010_2024["Year"].min(), trade_2010_2024["Year"].max())

(25417, 15)
(36699, 15)
2000 2009
2010 2024


2. Merge them into one dataframe:

In [164]:
fao_trade_df = pd.concat(
    [trade_2000_2009, trade_2010_2024],
    ignore_index=True
)

# Check the resulting shape
print(fao_trade_df.shape)

(62116, 15)


In [165]:
# Check for duplicated rows after merge
print("Duplicate full rows:", fao_trade_df.duplicated().sum())

Duplicate full rows: 0


3. Check how many countries, years and trade elements and items we have:

In [166]:
print(len(fao_trade_df["Area"].unique()), "countries")
print(len(fao_trade_df["Year"].unique()), "years")
print(len(fao_trade_df["Element"].unique()), "trade elements")
print(len(fao_trade_df["Item"].unique()), "trade items")

38 countries
25 years
2 trade elements
37 trade items


4. Check what combinations of Item + Element + Unit exist:

In [167]:
item_element_unit = fao_trade_df[["Item", "Element", "Unit"]].drop_duplicates().sort_values(["Item", "Element", "Unit"])
item_element_unit.head(10)

,Item,Element,Unit
0,Barley,Export quantity,t
1,Barley,Export value,1000 USD
20,Butter of cow milk,Export quantity,t
21,Butter of cow milk,Export value,1000 USD
40,Cattle,Export quantity,An
41,Cattle,Export value,1000 USD
1380,"Cheese from milk of sheep, fresh or processed",Export quantity,t
1381,"Cheese from milk of sheep, fresh or processed",Export value,1000 USD
60,Cheese from whole cow milk,Export quantity,t
61,Cheese from whole cow milk,Export value,1000 USD


We only have export data, and for each Item we have export quantity and export value in USD.

5. Create feature names reusing the previously created functions for simplifying items and elements names and using new similar function to simplify units names:

In [168]:
def simplify_unit_name(unit):
    unit = str(unit)

    replacements = {
        "1000 USD": "ths_usd",
        "1000 An": "ths_an",
    }

    for old, new in replacements.items():
        unit = unit.replace(old, new)

    return unit

fao_trade_df["feature_name"] = (
    fao_trade_df["Item"].apply(simplify_item_name).apply(clean_name)
    + "_"
    + fao_trade_df["Element"].apply(simplify_element_name).apply(clean_name)
    + "_"
    + fao_trade_df["Unit"].apply(simplify_unit_name).apply(clean_name)
)

# Check examples:
fao_trade_df[["Item", "Element", "Unit", "feature_name"]].drop_duplicates().sample(10)

,Item,Element,Unit,feature_name
21,Butter of cow milk,Export value,1000 USD,butter_of_cow_milk_export_value_ths_usd
5231,"Skim milk, condensed",Export value,1000 USD,skim_milk_condensed_export_value_ths_usd
101,"Cream, fresh",Export value,1000 USD,cream_fresh_export_value_ths_usd
1381,"Cheese from milk of sheep, fresh or processed",Export value,1000 USD,cheese_from_milk_of_sheep_export_value_ths_usd
61,Cheese from whole cow milk,Export value,1000 USD,cheese_from_whole_cow_milk_export_value_ths_usd
41,Cattle,Export value,1000 USD,cattle_export_value_ths_usd
461,Skim milk and whey powder,Export value,1000 USD,skim_milk_and_whey_powder_export_value_ths_usd
141,"Eggs, liquid",Export value,1000 USD,eggs_liquid_export_value_ths_usd
480,Skim milk of cows,Export quantity,t,skim_milk_of_cows_export_qty_t
581,Whole milk powder,Export value,1000 USD,whole_milk_powder_export_value_ths_usd


In [169]:
# Check duplicates before pivoting:
duplicates = fao_trade_df.duplicated(
    subset=["Area", "Year", "feature_name"],
    keep=False
)

print("Duplicate country-year-feature rows:", duplicates.sum())

Duplicate country-year-feature rows: 0


In [171]:
fao_trade_df_wide = fao_trade_df.pivot(
    index=["Area Code (M49)", "Area", "Year"],
    columns="feature_name",
    values="Value"
).reset_index()

# Check the result:
fao_trade_df_wide.head()

feature_name,Area Code (M49),Area,Year,barley_export_qty_t,barley_export_value_ths_usd,butter_of_cow_milk_export_qty_t,butter_of_cow_milk_export_value_ths_usd,cattle_export_qty_an,cattle_export_value_ths_usd,cheese_from_milk_of_sheep_export_qty_t,...,whole_milk_condensed_export_qty_t,whole_milk_condensed_export_value_ths_usd,whole_milk_powder_export_qty_t,whole_milk_powder_export_value_ths_usd,wool_degreased_or_carbonized_export_qty_t,wool_degreased_or_carbonized_export_value_ths_usd,wool_hair_waste_export_qty_t,wool_hair_waste_export_value_ths_usd,wool_shoddy_export_qty_t,wool_shoddy_export_value_ths_usd
0,32,Argentina,2000,36926.56,4991.0,6039.24,8822.0,14021.0,2180.0,NaN,...,303.0,314.0,97667.78,187366.0,6792.88,15167.0,2038.51,2867.0,NaN,NaN
1,32,Argentina,2001,201329.00,27320.0,2308.00,2992.0,79.0,11.0,NaN,...,59.0,88.0,84758.00,169917.0,7490.00,15282.0,2231.00,2762.0,NaN,NaN
2,32,Argentina,2002,112898.00,15903.0,3996.00,3877.0,0.0,0.0,NaN,...,79.0,67.0,136306.00,194629.0,7757.00,14355.0,3366.00,3868.0,NaN,NaN
3,32,Argentina,2003,66150.00,9731.0,558.00,878.0,1420.0,751.0,NaN,...,44.0,53.0,100403.00,180033.0,7984.00,17600.0,3969.00,4975.0,NaN,NaN
4,32,Argentina,2004,196665.00,31781.0,3900.00,7287.0,1429.0,558.0,NaN,...,29.0,38.0,176981.00,362539.0,5824.00,14256.0,4809.00,7645.0,NaN,NaN


Rename ID columns for MySQL:

In [172]:
fao_trade_df_wide = fao_trade_df_wide.rename(columns = {"Area Code (M49)": "country_code",
                                                      "Area": "country_name",
                                                      "Year": "year"})

In [173]:
fao_trade_df_wide.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 950 entries, 0 to 949
Data columns (total 76 columns):
 #   Column                                             Non-Null Count  Dtype  
---  ------                                             --------------  -----  
 0   country_code                                       950 non-null    int64  
 1   country_name                                       950 non-null    object 
 2   year                                               950 non-null    int64  
 3   barley_export_qty_t                                932 non-null    float64
 4   barley_export_value_ths_usd                        932 non-null    float64
 5   butter_of_cow_milk_export_qty_t                    932 non-null    float64
 6   butter_of_cow_milk_export_value_ths_usd            932 non-null    float64
 7   cattle_export_qty_an                               922 non-null    float64
 8   cattle_export_value_ths_usd                        922 non-null    float64
 9   cheese_fro

6. Store data frame to the database:

In [178]:
table_name = "faostat_crops_livestock_trade_export"
fao_trade_df_wide.to_sql(table_name, conn, index=False, schema=database, if_exists="replace")
# save changes to data to sql server
conn.commit()

#### Land use data set

1. Load Land Use dataset

For each of the selected countries this data set contains data on agricultural, crop, arable lands and permanent meadows and pastures areas.

In [54]:
fao_land_df = pd.read_csv("FAOSTAT_data_en_land_use.csv")
fao_land_df.head()

,Domain Code,Domain,Area Code (M49),Area,Element Code,Element,Item Code,Item,Year Code,Year,Unit,Value,Flag,Flag Description,Note
0,RL,Land Use,32,Argentina,5110,Area,6610,Agricultural land,2000,2000,1000 ha,128510.0,I,Value imputed by a receiving agency,NaN
1,RL,Land Use,32,Argentina,5110,Area,6610,Agricultural land,2001,2001,1000 ha,128606.0,I,Value imputed by a receiving agency,NaN
2,RL,Land Use,32,Argentina,5110,Area,6610,Agricultural land,2002,2002,1000 ha,128710.0,A,Official figure,NaN
3,RL,Land Use,32,Argentina,5110,Area,6610,Agricultural land,2003,2003,1000 ha,129103.3,I,Value imputed by a receiving agency,NaN
4,RL,Land Use,32,Argentina,5110,Area,6610,Agricultural land,2004,2004,1000 ha,129496.6,I,Value imputed by a receiving agency,NaN


In [55]:
fao_land_df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 3624 entries, 0 to 3623
Data columns (total 15 columns):
 #   Column            Non-Null Count  Dtype  
---  ------            --------------  -----  
 0   Domain Code       3624 non-null   object 
 1   Domain            3624 non-null   object 
 2   Area Code (M49)   3624 non-null   int64  
 3   Area              3624 non-null   object 
 4   Element Code      3624 non-null   int64  
 5   Element           3624 non-null   object 
 6   Item Code         3624 non-null   int64  
 7   Item              3624 non-null   object 
 8   Year Code         3624 non-null   int64  
 9   Year              3624 non-null   int64  
 10  Unit              3624 non-null   object 
 11  Value             3624 non-null   float64
 12  Flag              3624 non-null   object 
 13  Flag Description  3624 non-null   object 
 14  Note              0 non-null      float64
dtypes: float64(2), int64(5), object(8)
memory usage: 424.8+ KB


2. Check how many countries, years, items and units we have:

In [57]:
print(len(fao_land_df["Area"].unique()), "countries")
print(len(fao_land_df["Year"].unique()), "years")
print(len(fao_land_df["Item"].unique()), "items (land types)")
print(len(fao_land_df["Unit"].unique()), "units")

38 countries
24 years
4 items (land types)
1 units


In [59]:
print(fao_land_df["Year"].unique())

[2000 2001 2002 2003 2004 2005 2006 2007 2008 2009 2010 2011 2012 2013
 2014 2015 2016 2017 2018 2019 2020 2021 2022 2023]


Apparently, there is no data for 2024...

3. Create feature names

In [63]:
fao_land_df["feature_name"] = (
    fao_land_df["Item"].apply(clean_name)
    + "_ha"
)

# Check the result:
fao_land_df[["Item", "Element", "Unit", "feature_name"]].drop_duplicates()

,Item,Element,Unit,feature_name
0,Agricultural land,Area,1000 ha,agricultural_land_ha
24,Cropland,Area,1000 ha,cropland_ha
48,Arable land,Area,1000 ha,arable_land_ha
72,Permanent meadows and pastures,Area,1000 ha,permanent_meadows_and_pastures_ha


4. Pivot tha data frame

In [64]:
fao_land_use_df = fao_land_df.pivot(index=["Area Code (M49)", "Area", "Year"],
                                    columns="feature_name",
                                    values="Value"
                                   ).reset_index()

# check the result:
fao_land_use_df.head()

feature_name,Area Code (M49),Area,Year,agricultural_land_ha,arable_land_ha,cropland_ha,permanent_meadows_and_pastures_ha
0,32,Argentina,2000,128510.0,27640.0,28640.0,99870.0
1,32,Argentina,2001,128606.0,27746.0,28746.0,99860.0
2,32,Argentina,2002,128710.0,27862.0,28862.0,99848.0
3,32,Argentina,2003,129103.3,29828.2,30828.2,98275.1
4,32,Argentina,2004,129496.6,31794.4,32794.4,96702.2


5. Rename ID columns for MySQL:

In [65]:
fao_land_use_df = fao_land_use_df.rename(columns = {"Area Code (M49)": "country_code",
                                                    "Area": "country_name",
                                                    "Year": "year"})

# check the result:
fao_land_use_df.head()

feature_name,country_code,country_name,year,agricultural_land_ha,arable_land_ha,cropland_ha,permanent_meadows_and_pastures_ha
0,32,Argentina,2000,128510.0,27640.0,28640.0,99870.0
1,32,Argentina,2001,128606.0,27746.0,28746.0,99860.0
2,32,Argentina,2002,128710.0,27862.0,28862.0,99848.0
3,32,Argentina,2003,129103.3,29828.2,30828.2,98275.1
4,32,Argentina,2004,129496.6,31794.4,32794.4,96702.2


6. Store data frame to the database:

In [72]:
table_name = "faostat_land_use"
fao_land_use_df.to_sql(table_name, conn, index=False, schema=database, if_exists="replace")
# save changes to data to sql server
conn.commit()

Now that we uploaded all the raw data to the database, we can extract the features of interest from it, clean and engineer other necessary ones.

### Read data from MySQL

Since land use data set doesn't have data for 2024, we can filter the production and export data sets using sql-query.

In [316]:
prod_df = pd.read_sql(
    """
    SELECT *
    FROM faostat_crops_livestock_production
    WHERE year<2024;
    """,
    conn
)

export_df = pd.read_sql(
    """
    SELECT * 
    FROM faostat_crops_livestock_trade_export
    WHERE year<2024;
    """,
    conn
)

land_df = pd.read_sql(
    "SELECT * FROM faostat_land_use",
    conn
)

Check the shapes and first 10 features of retreived data sets:

In [317]:
print(prod_df.shape)
print(export_df.shape)
print(land_df.shape)

print(prod_df.columns[:10])
print(export_df.columns[:10])
print(land_df.columns[:10])

(912, 61)
(912, 76)
(912, 7)
Index(['country_code', 'country_name', 'year', 'barley_area_ha',
       'barley_prod_t', 'barley_yield_kg_per_ha', 'butter_of_cow_milk_prod_t',
       'cattle_stocks_an', 'cereals_n_e_c_area_ha', 'cereals_n_e_c_prod_t'],
      dtype='object')
Index(['country_code', 'country_name', 'year', 'barley_export_qty_t',
       'barley_export_value_ths_usd', 'butter_of_cow_milk_export_qty_t',
       'butter_of_cow_milk_export_value_ths_usd', 'cattle_export_qty_an',
       'cattle_export_value_ths_usd',
       'cheese_from_milk_of_sheep_export_qty_t'],
      dtype='object')
Index(['country_code', 'country_name', 'year', 'agricultural_land_ha',
       'arable_land_ha', 'cropland_ha', 'permanent_meadows_and_pastures_ha'],
      dtype='object')


For the purposes of clustering we are only going to use export value (USD) features, so we make a copy of `export_df` keeping only the `_export_value_ths_usd` columns.

In [318]:
# select export value columns only
export_value_cols = [col for col in export_df.columns if "_export_value_" in col]

# store the identification columns (countries + years) + totals
id_cols = ["country_code", "country_name", "year"]

# make a copy of export_df with only export value columns
export_value_df = export_df[id_cols + export_value_cols].copy()

# check the result
export_value_df.head()

,country_code,country_name,year,barley_export_value_ths_usd,butter_of_cow_milk_export_value_ths_usd,cattle_export_value_ths_usd,cheese_from_milk_of_sheep_export_value_ths_usd,cheese_from_whole_cow_milk_export_value_ths_usd,chickens_export_value_ths_usd,cream_fresh_export_value_ths_usd,...,skim_milk_of_cows_export_value_ths_usd,swine_per_pigs_export_value_ths_usd,vegetable_products_export_value_ths_usd,vegetables_frozen_export_value_ths_usd,wheat_export_value_ths_usd,whole_milk_condensed_export_value_ths_usd,whole_milk_powder_export_value_ths_usd,wool_degreased_or_carbonized_export_value_ths_usd,wool_hair_waste_export_value_ths_usd,wool_shoddy_export_value_ths_usd
0,32,Argentina,2000,4991.0,8822.0,2180.0,NaN,59414.0,2367.0,638.0,...,45.0,0.0,0.0,686.0,1218077.0,314.0,187366.0,15167.0,2867.0,NaN
1,32,Argentina,2001,27320.0,2992.0,11.0,NaN,49396.0,1871.0,408.0,...,35.0,0.0,2.0,624.0,1301552.0,88.0,169917.0,15282.0,2762.0,NaN
2,32,Argentina,2002,15903.0,3877.0,0.0,NaN,54183.0,328.0,108.0,...,0.0,4.0,16.0,632.0,1097362.0,67.0,194629.0,14355.0,3868.0,NaN
3,32,Argentina,2003,9731.0,878.0,751.0,NaN,53589.0,1.0,31.0,...,0.0,54.0,61.0,580.0,940518.0,53.0,180033.0,17600.0,4975.0,NaN
4,32,Argentina,2004,31781.0,7287.0,558.0,NaN,87383.0,0.0,98.0,...,26.0,24.0,0.0,926.0,1365480.0,38.0,362539.0,14256.0,7645.0,NaN


We also sum up the values of some columns referring to the same products.

For missing values, we use `.sum(axis=1, skipna=True, min_count=1)` which means:
- if all values are missing, result is NaN
- if one value exists, use the available value
- if some/all exist, sum them

In [319]:
# sum up the values for oats
export_value_df["oats_total_export_value_ths_usd"] = (
    export_value_df[["oats_export_value_ths_usd", "oats_rolled_export_value_ths_usd"]].sum(axis=1, skipna=True, min_count=1))

# drop unnecessary column(s)
export_value_df = export_value_df.drop(columns=["oats_export_value_ths_usd", "oats_rolled_export_value_ths_usd"])

Same for other features:

In [320]:
# all vegetables
export_value_df["vegetables_total_export_value_ths_usd"] = (
    export_value_df[["vegetable_products_export_value_ths_usd", "vegetables_frozen_export_value_ths_usd", 
             "other_vegetables_fresh_export_value_ths_usd"]].sum(axis=1, skipna=True, min_count=1))

export_value_df = export_value_df.drop(columns=["vegetable_products_export_value_ths_usd", "vegetables_frozen_export_value_ths_usd", 
                                "other_vegetables_fresh_export_value_ths_usd"])

# meat of cattle
export_value_df["meat_of_cattle_export_value_ths_usd"] = (
    export_value_df[["meat_of_cattle_boneless_export_value_ths_usd", "meat_of_cattle_with_the_bone_export_value_ths_usd"]].sum(axis=1, skipna=True, min_count=1))

export_value_df = export_value_df.drop(columns=["meat_of_cattle_boneless_export_value_ths_usd", "meat_of_cattle_with_the_bone_export_value_ths_usd"])

# meat of pig
export_value_df["meat_of_pig_export_value_ths_usd"] = (
    export_value_df[["meat_of_pig_boneless_export_value_ths_usd", "meat_of_pig_with_the_bone_export_value_ths_usd"]].sum(axis=1, skipna=True, min_count=1))

export_value_df = export_value_df.drop(columns=["meat_of_pig_boneless_export_value_ths_usd", "meat_of_pig_with_the_bone_export_value_ths_usd"])

# all milk
export_value_df["milk_export_value_ths_usd"] = (
    export_value_df[["raw_milk_of_cattle_export_value_ths_usd", "skim_milk_and_whey_powder_export_value_ths_usd", 
             "skim_milk_condensed_export_value_ths_usd", "skim_milk_of_cows_export_value_ths_usd",
             "whole_milk_condensed_export_value_ths_usd", "whole_milk_powder_export_value_ths_usd"]
           ].sum(axis=1, skipna=True, min_count=1))

export_value_df = export_value_df.drop(columns=["raw_milk_of_cattle_export_value_ths_usd", "skim_milk_and_whey_powder_export_value_ths_usd", 
             "skim_milk_condensed_export_value_ths_usd", "skim_milk_of_cows_export_value_ths_usd",
             "whole_milk_condensed_export_value_ths_usd", "whole_milk_powder_export_value_ths_usd"])

# all wool
export_value_df["wool_export_value_ths_usd"] = (
    export_value_df[["wool_degreased_or_carbonized_export_value_ths_usd", "wool_hair_waste_export_value_ths_usd", 
             "wool_shoddy_export_value_ths_usd"]].sum(axis=1, skipna=True, min_count=1))

export_value_df = export_value_df.drop(columns=["wool_degreased_or_carbonized_export_value_ths_usd", "wool_hair_waste_export_value_ths_usd", 
                                "wool_shoddy_export_value_ths_usd"])

# all eggs
export_value_df["hen_eggs_total_export_value_ths_usd"] = (
    export_value_df[["hen_eggs_in_shell_fresh_export_value_ths_usd", 
                     "eggs_liquid_export_value_ths_usd"]].sum(axis=1, skipna=True, min_count=1))

export_value_df = export_value_df.drop(columns=["hen_eggs_in_shell_fresh_export_value_ths_usd", "eggs_liquid_export_value_ths_usd"])

In [184]:
export_value_df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 912 entries, 0 to 911
Data columns (total 27 columns):
 #   Column                                             Non-Null Count  Dtype  
---  ------                                             --------------  -----  
 0   country_code                                       912 non-null    int64  
 1   country_name                                       912 non-null    object 
 2   year                                               912 non-null    int64  
 3   barley_export_value_ths_usd                        896 non-null    float64
 4   butter_of_cow_milk_export_value_ths_usd            895 non-null    float64
 5   cattle_export_value_ths_usd                        888 non-null    float64
 6   cheese_from_milk_of_sheep_export_value_ths_usd     366 non-null    float64
 7   cheese_from_whole_cow_milk_export_value_ths_usd    912 non-null    float64
 8   chickens_export_value_ths_usd                      879 non-null    float64
 9   cream_fres

`dairy_products_export_value_ths_usd` seem to have less non-null values (856) than some of the actual dairy products (e.g. `milk_export_value_ths_usd` (906) or `cheese_from_whole_cow_milk_export_value_ths_usd` (912)), so we calculate our own `dairy_products_total_export_value_ths_usd`:

In [321]:
# all dairy products
export_value_df["dairy_products_total_export_value_ths_usd"] = (
    export_value_df[["butter_of_cow_milk_export_value_ths_usd", "cheese_from_milk_of_sheep_export_value_ths_usd", 
                     "cheese_from_whole_cow_milk_export_value_ths_usd", "cream_fresh_export_value_ths_usd",
                     "milk_export_value_ths_usd"]].sum(axis=1, skipna=True, min_count=1))

export_value_df = export_value_df.drop(columns=["dairy_products_export_value_ths_usd"])

In [322]:
export_value_df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 912 entries, 0 to 911
Data columns (total 27 columns):
 #   Column                                             Non-Null Count  Dtype  
---  ------                                             --------------  -----  
 0   country_code                                       912 non-null    int64  
 1   country_name                                       912 non-null    object 
 2   year                                               912 non-null    int64  
 3   barley_export_value_ths_usd                        896 non-null    float64
 4   butter_of_cow_milk_export_value_ths_usd            895 non-null    float64
 5   cattle_export_value_ths_usd                        888 non-null    float64
 6   cheese_from_milk_of_sheep_export_value_ths_usd     366 non-null    float64
 7   cheese_from_whole_cow_milk_export_value_ths_usd    912 non-null    float64
 8   chickens_export_value_ths_usd                      879 non-null    float64
 9   cream_fres

#### Production data set

We want to do similar cleaning as we performed for export data.

Similarly, for the purposes of clustering: 
- For crops, we are only going to use `_yield_kg_per_ha` features as-is and `_area_ha` features to calculate the share of total cropland.

But before dropping main crops `_prod_t` features, we need to double-check whether FAOSTAT yield is missing even though prod area is not null:

In [333]:
# define crop names
crop_names = ["barley", "cereals_n_e_c", "oats", "potatoes", "pulses_total", "vegetables_primary", "wheat"]

crop_area_but_no_prod_yield_list = []

# for each crop
for crop in crop_names:
    # define column names
    area_col = f"{crop}_area_ha"
    prod_col = f"{crop}_prod_t"
    yield_col = f"{crop}_yield_kg_per_ha"
    
    # make a small copy with only the columns needed for this crop
    temp = prod_df[["country_code", "country_name", "year", area_col, prod_col, yield_col]].copy()
    
    # rename crop-specific columns to generic names so all crops can be combined
    temp = temp.rename(columns={
        area_col: "area_ha",
        prod_col: "prod_t",
        yield_col: "faostat_yield_kg_per_ha"
    })
    
    # keep crop name as a separate column
    temp["crop"] = crop

    # create a mask to find rows where
    # area exists and area is not 0 AND
    # production is missing or 0 AND/OR yield is missing or 0
    mask = (
        temp["area_ha"].notna()
        & (temp["area_ha"] != 0)
        & (
            temp["prod_t"].isna()
            | (temp["prod_t"] == 0)
            | temp["faostat_yield_kg_per_ha"].isna()
            | (temp["faostat_yield_kg_per_ha"] == 0)
        ))
    
    crop_area_but_no_prod_yield_list.append(temp[mask])

# combine each crop's data back together
crop_area_but_no_prod_yield_df = pd.concat(
    crop_area_but_no_prod_yield_list,
    ignore_index=True)

crop_area_but_no_prod_yield_df

,country_code,country_name,year,area_ha,prod_t,faostat_yield_kg_per_ha,crop
0,246,Finland,2018,3400.0,NaN,NaN,cereals_n_e_c
1,372,Ireland,2020,30.0,NaN,NaN,cereals_n_e_c
2,372,Ireland,2021,40.0,NaN,NaN,cereals_n_e_c
3,372,Ireland,2023,10.0,NaN,NaN,cereals_n_e_c
4,528,Netherlands (Kingdom of the),2018,920.0,NaN,NaN,cereals_n_e_c
5,528,Netherlands (Kingdom of the),2019,970.0,NaN,NaN,cereals_n_e_c
6,528,Netherlands (Kingdom of the),2020,1050.0,NaN,NaN,cereals_n_e_c
7,528,Netherlands (Kingdom of the),2023,960.0,NaN,NaN,cereals_n_e_c
8,528,Netherlands (Kingdom of the),2018,2070.0,NaN,NaN,pulses_total
9,528,Netherlands (Kingdom of the),2019,2800.0,NaN,NaN,pulses_total


Positive harvested area + missing production/yield should not be treated as zero yield.

If `area_ha` > 0, the country apparently has recorded harvested area for that crop. A yield of 0 would mean "nothing was produced from harvested area", which doesn't seem like a correct agricultural interpretation. More likely, it means production/yield was not reported, suppressed, unavailable, or the aggregate category is inconsistent.

In [330]:
# prod_df.to_csv("prod_df.csv")

In [143]:
prod_clean_df = prod_df.copy()
prod_clean_df = prod_clean_df.drop(columns=["barley_prod_t", "oats_prod_t", "cereals_n_e_c_prod_t", "potatoes_prod_t", 
                                            "pulses_total_prod_t", "vegetables_primary_prod_t", "wheat_prod_t"])

- For dairy products, we sum up some of the subcategories and create a total dairy production feature (like we did for export dataset):

In [144]:
# all milk
prod_clean_df["milk_prod_t"] = (
    prod_clean_df[["skim_milk_and_whey_powder_prod_t", "skim_milk_condensed_prod_t", 
                   "whole_milk_condensed_prod_t", "whole_milk_powder_prod_t"]].sum(axis=1, skipna=True, min_count=1))

prod_clean_df = prod_clean_df.drop(columns=["skim_milk_and_whey_powder_prod_t", "skim_milk_condensed_prod_t", 
                                            "whole_milk_condensed_prod_t", "whole_milk_powder_prod_t"])

# all cow milk cheese
prod_clean_df["cheese_from_cow_milk_prod_t"] = (prod_clean_df[["cheese_from_skimmed_cow_milk_prod_t", 
                                                               "cheese_from_whole_cow_milk_prod_t"]].sum(axis=1, skipna=True, min_count=1))

prod_clean_df = prod_clean_df.drop(columns=["cheese_from_skimmed_cow_milk_prod_t", "cheese_from_whole_cow_milk_prod_t"])

# all whey
prod_clean_df["whey_prod_t"] = (prod_clean_df[["whey_condensed_prod_t", "whey_dry_prod_t"]].sum(axis=1, skipna=True, min_count=1))

prod_clean_df = prod_clean_df.drop(columns=["whey_condensed_prod_t", "whey_dry_prod_t"])

# dairy products
prod_clean_df["dairy_prod_t"] = (
    prod_clean_df[["butter_of_cow_milk_prod_t", "cheese_from_milk_of_goats_prod_t", "cheese_from_milk_of_sheep_prod_t",
                   "cheese_from_cow_milk_prod_t", "cream_fresh_prod_t", "milk_prod_t", "whey_prod_t"]].sum(axis=1, skipna=True, min_count=1))

- From all the `eggs` features we are going to use only `hen_eggs_in_shell_fresh_prod_t`:

In [145]:
prod_clean_df = prod_clean_df.drop(columns=["hen_eggs_in_shell_fresh_laying_1000_an", "hen_eggs_in_shell_fresh_prod_1000_no", 
                                            "hen_eggs_in_shell_fresh_weight_g_per_an", "hen_eggs_in_shell_fresh_yield_no_per_an"])

- For `meat` features, we are only going to use main `_prod_t` features:

In [146]:
# choose the columns to drop: ones that start with "meat_of_" and end NOT with "_prod_t"
meat_cols_to_drop = [
    col for col in prod_clean_df.columns
    if col.startswith("meat_of_") and not col.endswith("_prod_t")
]
meat_cols_to_drop

['meat_of_cattle_animals_an',
 'meat_of_cattle_weight_kg_per_an',
 'meat_of_chickens_animals_1000_an',
 'meat_of_chickens_weight_g_per_an',
 'meat_of_goat_animals_an',
 'meat_of_goat_weight_kg_per_an',
 'meat_of_pig_animals_an',
 'meat_of_pig_weight_kg_per_an',
 'meat_of_sheep_animals_an',
 'meat_of_sheep_weight_kg_per_an']

In [147]:
# now drop them
prod_clean_df = prod_clean_df.drop(columns=meat_cols_to_drop)

In [148]:
prod_clean_df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 912 entries, 0 to 911
Data columns (total 36 columns):
 #   Column                              Non-Null Count  Dtype  
---  ------                              --------------  -----  
 0   country_code                        912 non-null    int64  
 1   country_name                        912 non-null    object 
 2   year                                912 non-null    int64  
 3   barley_area_ha                      912 non-null    float64
 4   barley_yield_kg_per_ha              906 non-null    float64
 5   butter_of_cow_milk_prod_t           854 non-null    float64
 6   cattle_stocks_an                    912 non-null    float64
 7   cereals_n_e_c_area_ha               517 non-null    float64
 8   cereals_n_e_c_yield_kg_per_ha       500 non-null    float64
 9   cheese_from_milk_of_goats_prod_t    478 non-null    float64
 10  cheese_from_milk_of_sheep_prod_t    494 non-null    float64
 11  chickens_stocks_1000_an             868 non-n

### Merge pre-cleaned data sets into one data frame

Use both `country_code` and `country_name` and `year` as the main keys. 

We would not rely only on country name, because country names can vary slightly between FAOSTAT datasets.

Check whether the selected merging keys match before merging:

In [187]:
prod_keys = set(zip(prod_clean_df["country_code"], prod_clean_df["country_name"], prod_clean_df["year"]))
export_keys = set(zip(export_value_df["country_code"], export_value_df["country_name"], export_value_df["year"]))
land_keys = set(zip(land_df["country_code"], land_df["country_name"], land_df["year"]))

print("prod - export:", len(prod_keys - export_keys))
print("export - prod:", len(export_keys - prod_keys))
print("prod - land:", len(prod_keys - land_keys))
print("land - prod:", len(land_keys - prod_keys))

prod - export: 0
export - prod: 0
prod - land: 0
land - prod: 0


They all are 0, meaning that they align perfectly. So we can proceed to merging and use `inner` join:

In [204]:
base_df = (
    prod_clean_df
    .merge(
        export_value_df,
        on=["country_code", "country_name", "year"],
        how="inner"
    )
    .merge(
        land_df,
        on=["country_code", "country_name", "year"],
        how="inner"
    )
)

# Check the shape of resulting merged data set, it should be [912 x (27+36+7-3*2=64)]
base_df.shape

(912, 64)

In [205]:
base_df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 912 entries, 0 to 911
Data columns (total 64 columns):
 #   Column                                             Non-Null Count  Dtype  
---  ------                                             --------------  -----  
 0   country_code                                       912 non-null    int64  
 1   country_name                                       912 non-null    object 
 2   year                                               912 non-null    int64  
 3   barley_area_ha                                     912 non-null    float64
 4   barley_yield_kg_per_ha                             906 non-null    float64
 5   butter_of_cow_milk_prod_t                          854 non-null    float64
 6   cattle_stocks_an                                   912 non-null    float64
 7   cereals_n_e_c_area_ha                              517 non-null    float64
 8   cereals_n_e_c_yield_kg_per_ha                      500 non-null    float64
 9   cheese_fro

### Note on raw FAOSTAT features

Most raw FAOSTAT production and trade variables are not directly suitable for clustering because they are heavily influenced by country size, agricultural land area, population, and overall economic scale. For example, total crop production in tonnes or export value in USD would make large countries appear different mainly because of scale, rather than because of agricultural structure. 

To make countries more comparable, raw variables were transformed into relative or intensity-based indicators, such as crop area as a share of cropland, livestock or animal product output per hectare of permanent meadows and pastures, export value shares, and land-structure indicators. Yield variables such as kg/ha are already normalised by area and were therefore retained more directly.

### Create land structure features

In [206]:
base_df["cropland_share_agri_land"] = (base_df["cropland_ha"] / base_df["agricultural_land_ha"])
base_df["pasture_share_agri_land"] = (base_df["permanent_meadows_and_pastures_ha"] / base_df["agricultural_land_ha"])

### Engineer crop area shares

Find crop area columns:

In [207]:
crop_area_cols = [col for col in base_df.columns if "area" in col and col.endswith("_ha")]

# Check which columns we have
crop_area_cols

['barley_area_ha',
 'cereals_n_e_c_area_ha',
 'oats_area_ha',
 'potatoes_area_ha',
 'pulses_total_area_ha',
 'vegetables_primary_area_ha',
 'wheat_area_ha']

Then divide by cropland (remembering that land data was in 1000 ha):

In [208]:
for col in crop_area_cols:
    new_col = col.replace("_area_ha", "_share_cropland")
    base_df[new_col] = base_df[col] / (base_df["cropland_ha"] * 1000)

# Check the result
crop_share_cols = [col for col in base_df.columns if col.endswith("_share_cropland")]
base_df[["country_name", "year"] + crop_share_cols].sample(10)

,country_name,year,barley_share_cropland,cereals_n_e_c_share_cropland,oats_share_cropland,potatoes_share_cropland,pulses_total_share_cropland,vegetables_primary_share_cropland,wheat_share_cropland
169,China,2001,0.005856,0.000093,0.002662,0.035901,0.028817,0.127022,0.187575
309,Estonia,2021,0.172369,0.008454,0.056752,0.004411,0.069475,0.001050,0.255319
69,Austria,2021,0.088907,0.007228,0.017520,0.016225,0.014355,0.011723,0.201273
791,Spain,2023,0.141045,0.000338,0.028107,0.003575,0.036454,0.018791,0.117891
561,Luxembourg,2009,0.147971,0.000537,0.021854,0.009537,0.004816,0.000647,0.218554
537,Lithuania,2009,0.133276,0.000096,0.030652,0.022389,0.022917,0.005755,0.240223
206,Croatia,2014,0.051836,0.002575,0.023746,0.011578,0.002705,0.038356,0.175339
836,Thailand,2020,0.003023,0.004258,NaN,0.000307,0.010133,0.016028,0.000058
39,Australia,2015,0.130829,NaN,0.026165,0.000937,0.063518,0.002215,0.359340
594,Malta,2018,0.000000,0.000000,0.000000,0.066474,0.000000,NaN,0.000000


In [210]:
base_df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 912 entries, 0 to 911
Data columns (total 73 columns):
 #   Column                                             Non-Null Count  Dtype  
---  ------                                             --------------  -----  
 0   country_code                                       912 non-null    int64  
 1   country_name                                       912 non-null    object 
 2   year                                               912 non-null    int64  
 3   barley_area_ha                                     912 non-null    float64
 4   barley_yield_kg_per_ha                             906 non-null    float64
 5   butter_of_cow_milk_prod_t                          854 non-null    float64
 6   cattle_stocks_an                                   912 non-null    float64
 7   cereals_n_e_c_area_ha                              517 non-null    float64
 8   cereals_n_e_c_yield_kg_per_ha                      500 non-null    float64
 9   cheese_fro

### Engineer animal stock intensity features

For pasture-based animals, we use permanent meadows and pastures (ha) as denominator:

In [211]:
base_df["cattle_stocks_per_pasture_ha"] = (
    base_df["cattle_stocks_an"] / base_df["permanent_meadows_and_pastures_ha"])

base_df["sheep_stocks_per_pasture_ha"] = (
    base_df["sheep_stocks_an"] / base_df["permanent_meadows_and_pastures_ha"])

base_df["goats_stocks_per_pasture_ha"] = (
    base_df["goats_stocks_an"] / base_df["permanent_meadows_and_pastures_ha"])

For chickens and pigs, pasture is not a good denominator, so we use agricultural land:

In [212]:
base_df["chickens_stocks_per_agri_land_ha"] = (
    base_df["chickens_stocks_1000_an"] * 1000 / base_df["agricultural_land_ha"])

base_df["swine_stocks_per_agri_land_ha"] = (
    base_df["swine_per_pigs_stocks_an"] / base_df["agricultural_land_ha"])

### Meat production intensity features

Same logic: raw tonnes are scale-dependent, so we convert to production intensity.

In [213]:
# For grazing animals, production is normalised by permanent meadows and pastures:
base_df["meat_of_cattle_prod_t_per_pasture_ha"] = (
    base_df["meat_of_cattle_prod_t"] / base_df["permanent_meadows_and_pastures_ha"])

base_df["meat_of_sheep_prod_t_per_pasture_ha"] = (
    base_df["meat_of_sheep_prod_t"] / base_df["permanent_meadows_and_pastures_ha"])

base_df["meat_of_goat_prod_t_per_pasture_ha"] = (
    base_df["meat_of_goat_prod_t"] / base_df["permanent_meadows_and_pastures_ha"])

# Chicken and pig meat are treated as non-pasture-based livestock production, so they
# are normalised by total agricultural land:
base_df["meat_of_chickens_prod_t_per_agri_land_ha"] = (
    base_df["meat_of_chickens_prod_t"] / base_df["agricultural_land_ha"])

base_df["meat_of_pig_prod_t_per_agri_land_ha"] = (
    base_df["meat_of_pig_prod_t"] / base_df["agricultural_land_ha"])

### Milk, dairy and egg production intensity

We use our already-cleaned aggregate dairy columns rather than all tiny dairy sub-products separately for now.

Milk and dairy production are important for Ireland, but raw production tonnes are also affected by country size. We convert them into intensity indicators. 

Milk and processed dairy are normalised by pasture area because they are closely linked to grazing/dairy livestock systems.

In [214]:
base_df["milk_prod_t_per_pasture_ha"] = (
    base_df["milk_prod_t"] / base_df["permanent_meadows_and_pastures_ha"])

base_df["dairy_prod_t_per_pasture_ha"] = (
    base_df["dairy_prod_t"] / base_df["permanent_meadows_and_pastures_ha"])

# Egg production is not pasture-based, so it is normalised by total agricultural land.
base_df["hen_eggs_prod_t_per_agri_land_ha"] = (
    base_df["hen_eggs_in_shell_fresh_prod_t"] / base_df["agricultural_land_ha"])

### Create export value shares

Export values in USD are not used directly because large countries naturally export more in absolute terms. 

Instead, each product's export value is converted into a share of the country's total Crops & Livestock export value. This captures export specialisation.

In [215]:
# rename total Crops & Livestock export value feature
# so that it's not in the colums to calculate share for
base_df = base_df.rename(columns = {"crops_and_livestock_products_export_value_ths_usd": "crops_and_livestock_products_export_total_ths_usd"})

In [216]:
# select export value features
export_value_cols = [
    col for col in base_df.columns
    if col.endswith("_export_value_ths_usd")]

export_value_cols

['barley_export_value_ths_usd',
 'butter_of_cow_milk_export_value_ths_usd',
 'cattle_export_value_ths_usd',
 'cheese_from_milk_of_sheep_export_value_ths_usd',
 'cheese_from_whole_cow_milk_export_value_ths_usd',
 'chickens_export_value_ths_usd',
 'cream_fresh_export_value_ths_usd',
 'goats_export_value_ths_usd',
 'meat_of_chickens_export_value_ths_usd',
 'meat_of_goat_export_value_ths_usd',
 'meat_of_sheep_export_value_ths_usd',
 'potatoes_export_value_ths_usd',
 'sheep_export_value_ths_usd',
 'swine_per_pigs_export_value_ths_usd',
 'wheat_export_value_ths_usd',
 'oats_total_export_value_ths_usd',
 'vegetables_total_export_value_ths_usd',
 'meat_of_cattle_export_value_ths_usd',
 'meat_of_pig_export_value_ths_usd',
 'milk_export_value_ths_usd',
 'wool_export_value_ths_usd',
 'hen_eggs_total_export_value_ths_usd',
 'dairy_products_total_export_value_ths_usd']

In [217]:
# for each of selected columns calculate share
for col in export_value_cols:
    new_col = col.replace("_export_value_ths_usd", "_export_value_share")
    base_df[new_col] = base_df[col] / base_df["crops_and_livestock_products_export_total_ths_usd"]

### Define final feature groups

Now we define what will actually go into clustering

In [235]:
# Land structure features describe the broad agricultural land profile of each country
land_structure_cols = [
    "cropland_share_agri_land",
    "pasture_share_agri_land"
]

In [236]:
# Crop shares describe how each country's cropland is allocated across selected crops
crop_share_cols = [col for col in base_df.columns if "_share_cropland" in col]
crop_share_cols

['barley_share_cropland',
 'cereals_n_e_c_share_cropland',
 'oats_share_cropland',
 'potatoes_share_cropland',
 'pulses_total_share_cropland',
 'vegetables_primary_share_cropland',
 'wheat_share_cropland']

In [288]:
# Crop yields are already area-normalised, so they can be used directly after scaling
crop_yield_cols = [col for col in base_df.columns if "_yield_" in col and "_kg_per_ha" in col]
crop_yield_cols

['barley_yield_kg_per_ha',
 'cereals_n_e_c_yield_kg_per_ha',
 'oats_yield_kg_per_ha',
 'potatoes_yield_kg_per_ha',
 'pulses_total_yield_kg_per_ha',
 'vegetables_primary_yield_kg_per_ha',
 'wheat_yield_kg_per_ha']

In [238]:
# Animal features are intensity-based, not raw counts or tonnes
animal_intensity_cols = [col for col in base_df.columns if col.endswith("_per_pasture_ha") 
                         or col.endswith("_per_agri_land_ha")]
animal_intensity_cols

['cattle_stocks_per_pasture_ha',
 'sheep_stocks_per_pasture_ha',
 'goats_stocks_per_pasture_ha',
 'chickens_stocks_per_agri_land_ha',
 'swine_stocks_per_agri_land_ha',
 'meat_of_cattle_prod_t_per_pasture_ha',
 'meat_of_sheep_prod_t_per_pasture_ha',
 'meat_of_goat_prod_t_per_pasture_ha',
 'meat_of_chickens_prod_t_per_agri_land_ha',
 'meat_of_pig_prod_t_per_agri_land_ha',
 'milk_prod_t_per_pasture_ha',
 'dairy_prod_t_per_pasture_ha',
 'hen_eggs_prod_t_per_agri_land_ha']

In [239]:
# Export features describe product-level export specialisation
# but remember, we only want milk and total dairy products for now
# (so we exclude explicitly features having "butter" or "cheese" or "cream" in their names)
export_share_cols = [col for col in base_df.columns if col.endswith("_export_value_share")
                     and not ("butter_" in col or "cheese_" in col or "cream_" in col)]
export_share_cols

['barley_export_value_share',
 'cattle_export_value_share',
 'chickens_export_value_share',
 'goats_export_value_share',
 'meat_of_chickens_export_value_share',
 'meat_of_goat_export_value_share',
 'meat_of_sheep_export_value_share',
 'potatoes_export_value_share',
 'sheep_export_value_share',
 'swine_per_pigs_export_value_share',
 'wheat_export_value_share',
 'oats_total_export_value_share',
 'vegetables_total_export_value_share',
 'meat_of_cattle_export_value_share',
 'meat_of_pig_export_value_share',
 'milk_export_value_share',
 'wool_export_value_share',
 'hen_eggs_total_export_value_share',
 'dairy_products_total_export_value_share']

Combine the features

In [243]:
feature_cols = (
    land_structure_cols
    + crop_share_cols
    + crop_yield_cols
    + animal_intensity_cols
    + export_share_cols
)

len(feature_cols)

48

### Create clustering dataframe

This is where we drop raw features from the modelling dataset.

In [246]:
# Quick check before copying features of interest from base_df
# Ratios could create infinity if a denominator was zero
# Replace infinities with NaN so they are handled consistently as missing values:
base_df = base_df.replace([np.inf, -np.inf], np.nan)

In [247]:
# base_df remains the full working dataset with raw and engineered columns
# cluster_df contains only identifiers and engineered/comparable features
cluster_df = base_df[["country_code", "country_name", "year"] + feature_cols].copy()
cluster_df.shape

(912, 51)

Now check missing values:

In [262]:
# Calculate the percentage of missing values in each feature column, 
# then sort features from most missing to least missing
missing_by_feature = (
    cluster_df[feature_cols]
    .isna()                         # check where values are missing: True=1, False=0
    .mean()                         # missing-value share per column: mean of True/False values gives the proportion of missing values
    .sort_values(ascending=False)   # sort from most missing to least missing
)

# display only columns with missing values
missing_by_feature[missing_by_feature > 0]

cereals_n_e_c_yield_kg_per_ha               0.451754
cereals_n_e_c_share_cropland                0.433114
meat_of_goat_export_value_share             0.270833
meat_of_goat_prod_t_per_pasture_ha          0.226974
goats_export_value_share                    0.163377
hen_eggs_prod_t_per_agri_land_ha            0.096491
goats_stocks_per_pasture_ha                 0.083333
oats_yield_kg_per_ha                        0.078947
milk_prod_t_per_pasture_ha                  0.075658
oats_share_cropland                         0.072368
sheep_export_value_share                    0.072368
swine_per_pigs_export_value_share           0.064693
meat_of_cattle_prod_t_per_pasture_ha        0.052632
chickens_stocks_per_agri_land_ha            0.048246
chickens_export_value_share                 0.036184
wool_export_value_share                     0.030702
sheep_stocks_per_pasture_ha                 0.030702
pasture_share_agri_land                     0.026316
dairy_prod_t_per_pasture_ha                 0.

Check missingness by country-year:

In [251]:
# This shows whether some country-years have many missing features
cluster_df["missing_feature_count"] = cluster_df[feature_cols].isna().sum(axis=1)
cluster_df["missing_feature_share"] = cluster_df["missing_feature_count"] / len(feature_cols)

cluster_df[
    ["country_name", "year", "missing_feature_count", "missing_feature_share"]
].sort_values("missing_feature_share", ascending=False).head(20)

,country_name,year,missing_feature_count,missing_feature_share
597,Malta,2021,29,0.604167
599,Malta,2023,28,0.583333
598,Malta,2022,28,0.583333
596,Malta,2020,28,0.583333
595,Malta,2019,27,0.562500
594,Malta,2018,27,0.562500
591,Malta,2015,25,0.520833
592,Malta,2016,24,0.500000
593,Malta,2017,22,0.458333
590,Malta,2014,21,0.437500


Check Ireland specifically:

In [252]:
cluster_df.loc[
    cluster_df["country_name"] == "Ireland",
    ["country_name", "year", "missing_feature_count", "missing_feature_share"]
]

,country_name,year,missing_feature_count,missing_feature_share
456,Ireland,2000,1,0.020833
457,Ireland,2001,1,0.020833
458,Ireland,2002,1,0.020833
459,Ireland,2003,1,0.020833
460,Ireland,2004,1,0.020833
461,Ireland,2005,1,0.020833
462,Ireland,2006,1,0.020833
463,Ireland,2007,1,0.020833
464,Ireland,2008,1,0.020833
465,Ireland,2009,1,0.020833


## Aggregate recent years' data per country

We aggregate recent years before clustering because the goal is to compare countries, not individual country-year observations. 

If we clustered the data "as-is", each country would appear multiple times, and the clustering could be influenced by annual shocks, weather effects, price changes, or temporary trade fluctuations rather than the country's typical agricultural profile. Multiple rows from the same country are also not truly independent observations. 

Averaging over recent years (let's say 2015+) gives one, more stable profile per country while still reflecting the current structure of production, land use and exports.

In [356]:
# get only the recent years
recent_cluster_df = cluster_df[
    cluster_df["year"].between(2015, 2024)
].copy()

# save recent years for future use
recent_cluster_df.drop(columns=["missing_feature_count", "missing_feature_share"]).to_csv("recent_cluster_df.csv", index=False)

# drop year (and additional columns created for missing value checks)
# calculate average values for each feature by country
# pandas mean skips NaN values by default, so one missing year does not remove the whole country from the analysis
country_cluster_df = (
    recent_cluster_df
    .drop(columns=["year", "missing_feature_count", "missing_feature_share"])
    .groupby(["country_code", "country_name"], as_index=False)
    .mean(numeric_only=True)
)

#### Check missingness again after aggregation

This is the more important missingness check for clustering:

In [335]:
# select feature columns (that are not country_code and country_name)
country_feature_cols = [col for col in country_cluster_df.columns if col not in ["country_code", "country_name"]]

# check missing values the same way as before
missing_after_aggregation = (
    country_cluster_df[country_feature_cols]
    .isna()
    .mean()
    .sort_values(ascending=False)
)

missing_after_aggregation[missing_after_aggregation > 0]

cereals_n_e_c_yield_kg_per_ha           0.342105
cereals_n_e_c_share_cropland            0.289474
meat_of_goat_export_value_share         0.105263
meat_of_goat_prod_t_per_pasture_ha      0.105263
oats_yield_kg_per_ha                    0.078947
goats_export_value_share                0.078947
oats_share_cropland                     0.052632
wool_export_value_share                 0.052632
meat_of_cattle_prod_t_per_pasture_ha    0.052632
swine_per_pigs_export_value_share       0.052632
meat_of_sheep_prod_t_per_pasture_ha     0.026316
sheep_export_value_share                0.026316
pasture_share_agri_land                 0.026316
sheep_stocks_per_pasture_ha             0.026316
milk_prod_t_per_pasture_ha              0.026316
dairy_prod_t_per_pasture_ha             0.026316
goats_stocks_per_pasture_ha             0.026316
cattle_export_value_share               0.026316
cattle_stocks_per_pasture_ha            0.026316
chickens_export_value_share             0.026316
oats_total_export_va

Check country-level missingness:

In [336]:
country_cluster_df["missing_feature_count"] = (
    country_cluster_df[country_feature_cols].isna().sum(axis=1)
)

country_cluster_df["missing_feature_share"] = (
    country_cluster_df["missing_feature_count"] / len(country_feature_cols)
)

country_cluster_df[
    ["country_name", "missing_feature_count", "missing_feature_share"]
].sort_values("missing_feature_share", ascending=False)

,country_name,missing_feature_count,missing_feature_share
24,Malta,18,0.375000
37,Uruguay,7,0.145833
18,India,5,0.104167
9,Cyprus,4,0.083333
35,United Kingdom of Great Britain and Northern I...,3,0.062500
6,Canada,3,0.062500
1,Australia,2,0.041667
15,Germany,2,0.041667
13,Finland,2,0.041667
11,Denmark,2,0.041667


Malta has substantially higher missingness than all other countries after aggregation to the recent years country-level dataset. 

Since almost 40% of its clustering features are missing, including Malta would require heavy imputation and could distort the clustering results.

Therefore, Malta is removed before final feature filtering and imputation.

In [337]:
country_cluster_df = country_cluster_df[country_cluster_df["country_name"] != "Malta"].copy()

Recalculate missingness after dropping Malta:

In [338]:
country_feature_cols = [
    col for col in country_cluster_df.columns
    if col not in [
        "country_code",
        "country_name",
        "missing_feature_count",
        "missing_feature_share"
    ]
]

missing_after_dropping_malta = (
    country_cluster_df[country_feature_cols]
    .isna()
    .mean()
    .sort_values(ascending=False)
)

missing_after_dropping_malta[missing_after_dropping_malta>0]

cereals_n_e_c_yield_kg_per_ha           0.324324
cereals_n_e_c_share_cropland            0.297297
meat_of_goat_export_value_share         0.081081
meat_of_goat_prod_t_per_pasture_ha      0.081081
oats_share_cropland                     0.054054
goats_export_value_share                0.054054
oats_yield_kg_per_ha                    0.054054
swine_per_pigs_export_value_share       0.027027
oats_total_export_value_share           0.027027
chickens_export_value_share             0.027027
meat_of_cattle_prod_t_per_pasture_ha    0.027027
wool_export_value_share                 0.027027
dtype: float64

#### We keep only features that are available for at least 70% of countries

With a small number of countries, very sparse features can distort clustering

In [339]:
good_features = (
    missing_after_dropping_malta[missing_after_aggregation <= 0.30]
    .index
    .tolist()
)
len(good_features)

47

In [340]:
# create the final modelling matrix:
country_cluster_clean = country_cluster_df[["country_code", "country_name"] + good_features].copy()

In [341]:
# country_cluster_clean = country_cluster_df

In [342]:
# look only at countries and features with any missing values in country_cluster_df
# separate id columns
id_cols = ["country_code", "country_name"]

# select only feature columns
feature_cols = [col for col in country_cluster_clean.columns if col not in id_cols 
                and col not in ["missing_feature_count", "missing_feature_share"]]

# features that have at least one missing value
features_with_missing = (
    country_cluster_clean[feature_cols]
    .columns[country_cluster_clean[feature_cols].isna().any()]
    .tolist()
)

# countries that have at least one missing value in those features
countries_with_missing = country_cluster_clean[
    country_cluster_clean[features_with_missing].isna().any(axis=1)]

# show only relevant countries and relevant missing-feature columns
missing_view = countries_with_missing[
    id_cols + features_with_missing]

missing_view

,country_code,country_name,cereals_n_e_c_share_cropland,meat_of_goat_export_value_share,meat_of_goat_prod_t_per_pasture_ha,oats_share_cropland,goats_export_value_share,oats_yield_kg_per_ha,swine_per_pigs_export_value_share,oats_total_export_value_share,chickens_export_value_share,meat_of_cattle_prod_t_per_pasture_ha,wool_export_value_share
1,36,Australia,NaN,4.228594e-03,0.084359,0.026762,1.246232e-04,1631.344444,6.231316e-07,0.003451,0.000107,6.772202,0.004284
4,76,Brazil,NaN,1.566682e-07,0.204840,0.006914,3.016865e-09,2030.877778,4.332203e-05,0.000020,0.000882,45.909196,0.000002
6,124,Canada,NaN,1.756353e-06,NaN,0.028667,3.548786e-06,3350.300000,7.778953e-03,0.012648,0.000305,67.113654,0.000012
9,196,Cyprus,NaN,2.338067e-04,1220.508768,0.002373,2.166708e-03,1468.300000,0.000000e+00,NaN,0.000004,2865.334672,NaN
11,208,Denmark,NaN,3.106038e-06,0.000000,0.025967,0.000000e+00,4913.011111,6.628642e-02,0.000918,0.005629,545.199910,0.000009
12,233,Estonia,0.006950,NaN,0.044099,0.050565,2.489402e-06,2446.544444,1.843852e-03,0.012079,0.000076,35.962797,0.000243
13,246,Finland,0.000913,NaN,0.348485,0.133096,NaN,3447.411111,1.673506e-03,0.047398,0.001459,4004.228023,0.000026
15,276,Germany,NaN,5.227932e-06,0.061678,0.011843,9.871637e-07,4304.822222,2.564894e-03,0.001547,0.007306,231.198988,0.000173
18,356,India,NaN,3.795434e-06,114.955788,NaN,4.445289e-04,NaN,7.304398e-06,0.000048,0.000011,NaN,0.000289
33,752,Sweden,NaN,8.955212e-07,0.029009,0.061296,1.994820e-08,4046.988889,2.007574e-05,0.008431,0.002328,302.062601,0.000009


## Handling missing values

Missing values are handled according to feature meaning rather than applying one blanket imputation rule. 
- For export shares, crop shares, and livestock/product intensity features, missing values often indicate that the country has no recorded export, crop area, stock, or production for that item, so these are set to 0.
- Crop yields are treated more carefully: if the crop share is 0 and yield is missing, the yield is set to 0 because the crop is not part of that country's production system.
- Only remaining missing values (if any), which are more likely to represent unavailable data, will be imputed using the feature median before scaling and clustering.

In [343]:
# For export shares
# If a country has no recorded export value for a selected product, it is reasonable to treat the share as 0, not median:
country_cluster_clean[export_share_cols] = (country_cluster_clean[export_share_cols].fillna(0))

In [344]:
# For crop shares
# Same logic: if a country has no recorded area for a crop, its share of cropland can reasonably be treated as 0
country_cluster_clean[crop_share_cols] = (country_cluster_clean[crop_share_cols].fillna(0))

In [345]:
# For animal/product intensity features
# Missing here means no meaningful production/stock record for that country
# For clustering, using 0 would be more interpretable than median
country_cluster_clean[animal_intensity_cols] = (country_cluster_clean[animal_intensity_cols].fillna(0))

In [346]:
# Crop yield is different
# If a country does not grow a crop, the yield is not really 'missing' — it is not applicable
# So for missing yield where cropland share is zero, we can set yield to 0
for yield_col in crop_yield_cols:
    share_col = yield_col.replace("_yield_kg_per_ha", "_share_cropland")  # define _share_cropland column name for corresponding crop
    
    if yield_col in country_cluster_clean.columns and share_col in country_cluster_clean.columns:  # check that both (yield and share) columns exist
        country_cluster_clean.loc[
            # if crop share is 0 and yield is missing
            country_cluster_clean[yield_col].isna() & (country_cluster_clean[share_col] == 0), 
            # we interpret this as the crop not being part of that country's system (set yield to 0)
            yield_col] = 0

In [348]:
crop_yield_cols_ = [
    "barley_yield_kg_per_ha",
    "oats_yield_kg_per_ha",
    "potatoes_yield_kg_per_ha",
    "pulses_total_yield_kg_per_ha",
    "vegetables_primary_yield_kg_per_ha",
    "wheat_yield_kg_per_ha"
]
country_cluster_clean[["country_name"]+crop_yield_cols_]

,country_name,barley_yield_kg_per_ha,oats_yield_kg_per_ha,potatoes_yield_kg_per_ha,pulses_total_yield_kg_per_ha,vegetables_primary_yield_kg_per_ha,wheat_yield_kg_per_ha
0,Argentina,3705.222222,2138.622222,33928.622222,1393.188889,21248.477778,2975.033333
1,Australia,2555.155556,1631.344444,39758.166667,1467.111111,26408.855556,2320.988889
2,Austria,5917.777778,3810.355556,31453.311111,2263.233333,35278.400000,5626.611111
3,Belgium,7925.355556,5044.988889,41184.211111,4234.622222,32930.488889,8607.244444
4,Brazil,3353.122222,2030.877778,31663.255556,1084.422222,24857.800000,2667.322222
5,Bulgaria,4733.500000,2294.277778,17867.844444,1815.922222,20248.133333,5100.966667
6,Canada,3514.733333,3350.300000,39751.222222,1883.211111,24677.166667,3228.455556
7,China,3606.533333,3415.566667,19485.788889,1813.222222,24840.077778,5611.733333
8,Croatia,4795.000000,3064.044444,17265.633333,1681.677778,32712.662500,5698.500000
9,Cyprus,1804.033333,1468.300000,23403.377778,1522.155556,30247.222222,2120.755556


In [349]:
# Look if anything left missing
country_cluster_clean[feature_cols].isna().sum().sum()

np.int64(0)

Nope. Great!

Save the resulting data set into csv for future use:

In [353]:
country_cluster_clean.to_csv("country_cluster_clean.csv", index=False)

In [351]:
country_cluster_clean.shape

(37, 49)

In [352]:
country_cluster_clean

,country_code,country_name,cereals_n_e_c_share_cropland,meat_of_goat_export_value_share,meat_of_goat_prod_t_per_pasture_ha,oats_share_cropland,goats_export_value_share,oats_yield_kg_per_ha,swine_per_pigs_export_value_share,oats_total_export_value_share,...,vegetables_primary_yield_kg_per_ha,wheat_yield_kg_per_ha,cattle_stocks_per_pasture_ha,sheep_stocks_per_pasture_ha,goats_stocks_per_pasture_ha,chickens_stocks_per_agri_land_ha,swine_stocks_per_agri_land_ha,meat_of_sheep_prod_t_per_pasture_ha,pasture_share_agri_land,dairy_products_total_export_value_share
0,32,Argentina,0.000623,8.366176e-06,0.100579,0.006409,8.600989e-08,2138.622222,2.706893e-06,0.000024,...,21248.477778,2975.033333,711.147315,185.659348,59.498062,980.885241,44.965460,0.647403,0.642700,0.020170
1,36,Australia,0.000000,4.228594e-03,0.084359,0.026762,1.246232e-04,1631.344444,6.231316e-07,0.003451,...,26408.855556,2320.988889,78.669116,210.390112,11.248691,283.131186,6.735344,2.187876,0.913046,0.042160
2,40,Austria,0.007498,2.484246e-07,0.552426,0.015415,6.655220e-05,3810.355556,2.601049e-04,0.000565,...,35278.400000,5626.611111,1528.341095,311.590072,72.072636,6392.821507,1044.809756,5.256979,0.470803,0.068638
3,56,Belgium,0.003205,8.021801e-06,0.731195,0.004173,6.513144e-06,5044.988889,3.346540e-03,0.000455,...,32930.488889,8607.244444,4998.419831,213.700911,106.978633,34974.440248,4469.953178,5.390249,0.351128,0.068601
4,76,Brazil,0.000000,1.566682e-07,0.204840,0.006914,3.016865e-09,2030.877778,4.332203e-05,0.000020,...,24857.800000,2667.322222,1278.549534,114.709228,65.149872,6176.974857,175.779599,0.568070,0.732121,0.001173
5,100,Bulgaria,0.000805,1.963126e-05,1.121299,0.003393,1.452371e-05,2294.277778,5.871501e-04,0.000343,...,20248.133333,5100.966667,405.234714,903.577437,170.254079,3371.548560,122.203744,5.867347,0.277299,0.021332
6,124,Canada,0.000000,1.756353e-06,0.000000,0.028667,3.548786e-06,3350.300000,7.778953e-03,0.012648,...,24677.166667,3228.455556,610.141084,43.480356,1.586515,2956.200661,242.132256,0.883712,0.329130,0.002939
7,159,China,0.000142,2.209215e-04,6.097021,0.001196,9.890234e-06,3415.566667,6.648837e-03,0.000116,...,24840.077778,5611.733333,169.032682,442.468463,347.200584,9929.446319,830.149116,6.341317,0.752227,0.009446
8,191,Croatia,0.001524,7.722002e-05,0.930780,0.020850,3.231252e-05,3064.044444,2.051210e-02,0.000682,...,32712.662500,5698.500000,748.382860,1100.714111,136.989285,7153.358181,690.910206,9.440220,0.383478,0.020076
9,196,Cyprus,0.000000,2.338067e-04,1220.508768,0.002373,2.166708e-03,1468.300000,0.000000e+00,0.000000,...,30247.222222,2120.755556,39956.858404,177665.503588,137097.299137,30795.222668,2816.032675,1647.662609,0.015112,0.544071
